# Ночной прогон: только русский, строгие 3/3
Первый проход EN определяет язык. Только ru без неопределённости получает ещё два прохода. Кандидаты не являются человеческими эталонами. Код решения общий с Airflow.


In [ ]:
import json, hashlib
import pandas as pd
from IPython.display import Audio, display
from lalm.annotation.inspection import open_snapshot
from lalm.annotation.contract import TRIAD_VERSION, triad_decision
s3, snapshot, cache, db = open_snapshot()
print('Снимок:', snapshot['snapshot_at'])
passes = db.sql('SELECT * FROM passes').df()
new = passes[passes.batch_id.fillna('').str.startswith(TRIAD_VERSION+'-')]
records=[]
for (batch, sample), group in new.groupby(['batch_id','sample_id']):
    results={r.prompt_variant: {'verbatim_text':r.gemini_text,'language':r.language or '',
                'uncertain_spans':json.loads(r.uncertain_spans) if isinstance(r.uncertain_spans,str) else []}
             for r in group.itertuples() if isinstance(r.gemini_text,str)}
    old=passes[(passes.sample_id==sample)&(passes.batch_id!=batch)].gemini_text.dropna().tolist()
    decision=triad_decision(results,old)
    if decision=='waiting_remaining' and group.original_annotation_status.eq('not_linked_or_not_available').all():
        decision='missing_source_annotation'
    if ((group.processing_state.isin(['NEEDS_REVIEW','PERMANENT_FAILED'])) & group.gemini_text.isna()).any():
        decision='rejected_failed_pass'
    records.append({'batch_id':batch,'sample_id':sample,'decision':decision})
status=pd.DataFrame(records,columns=['batch_id','sample_id','decision'])
display(status.groupby('decision').size().rename('examples').to_frame())
display(new.groupby(['account_id','processing_state'],dropna=False).size().rename('requests').to_frame())


In [ ]:
# Для проверки precision смотрим кандидатов; для диагностики можно сменить фильтр.
selection = 'candidate_3of3'
selected = status[status.decision==selection].head(30).reset_index(drop=True)
display(selected)
for row in selected.head(5).itertuples():
    group=new[(new.sample_id==row.sample_id)&(new.batch_id==row.batch_id)]
    display(group[['prompt_variant','original_annotation','gemini_text','language','error_code']])
    audio=group.iloc[0]
    data=s3.get_object(Bucket=audio.audio_bucket,Key=audio.audio_key)['Body'].read()
    assert hashlib.sha256(data).hexdigest()==audio.audio_sha256
    display(Audio(data))
